# Prepare PhysioNet A+B Train / C Test Data for Logistic Regression and XGBoost

This notebook prepares leakage-safe feature matrices for binary ICU mortality prediction.

## Preprocessing and imputation strategy

The training dataset is created by combining **Set A + Set B**, and **Set C** is held out as the final test dataset.

The following columns are removed before modeling because they are identifiers, targets, severity-score summaries, or outcome-related variables:

- `RecordID`
- `in_hospital_death`
- `SAPS-I`
- `SOFA`
- `Length_of_stay`
- `Survival`

Missingness filtering is learned from the **A+B training data only** and then applied unchanged to Set C. This avoids using information from the held-out test set during preprocessing.

For clinical variables that have a corresponding `*_count` feature, a value of `count = 0` means that the original clinical variable was not measured for that patient. Therefore, this notebook treats `count = 0` as measurement-level missingness. If at least **50%** of A+B training patients have `count = 0` or missing count for a clinical variable, the entire feature group for that variable is dropped. For example, if `Lactate_count` is zero or missing in at least 50% of training patients, then features such as `Lactate_mean`, `Lactate_min`, `Lactate_max`, `Lactate_first`, `Lactate_last`, `Lactate_count`, and `Lactate_missing` are removed together.

After this group-level filtering, any remaining individual feature column with at least **50%** missing values in A+B training data is also removed.

For imputation, remaining `*_count` columns are filled with `0`, because zero means no measurement was observed. All other remaining numeric features are mean-imputed using means estimated from the **A+B training data only**. The same trained imputation values are then applied to Set C.

The output files are saved in an `output/` folder located at the same level as `code/` and `data/`.


In [1]:
# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [2]:
# ============================================================
# 1. Path setup
# ------------------------------------------------------------
# Expected project structure:
#
# project_root/
#   code/      <- run this notebook here
#   data/      <- input CSV files here
#   output/    <- output files will be saved here
# ============================================================

CODE_DIR = Path.cwd()
PROJECT_ROOT = CODE_DIR.parent if CODE_DIR.name.lower() == "code" else CODE_DIR
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CODE_DIR     = {CODE_DIR}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_DIR     = {DATA_DIR}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")


CODE_DIR     = c:\Users\junse\Documents\research\IUBDC 2026\code
PROJECT_ROOT = c:\Users\junse\Documents\research\IUBDC 2026
DATA_DIR     = c:\Users\junse\Documents\research\IUBDC 2026\data
OUTPUT_DIR   = c:\Users\junse\Documents\research\IUBDC 2026\output


In [3]:
# ============================================================
# 2. Load input files
# ============================================================

SET_A_FILE = DATA_DIR / "set_a_ml_ready_with_outcomes.csv"
SET_B_FILE = DATA_DIR / "set_b_ml_ready_with_outcomes.csv"
SET_C_FILE = DATA_DIR / "set_c_ml_ready_with_outcomes.csv"

required_files = [SET_A_FILE, SET_B_FILE, SET_C_FILE]
missing_files = [str(p) for p in required_files if not p.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" +
        "\n".join(missing_files) +
        "\n\nMake sure this notebook is run from the code/ folder and the CSV files are in ../data/."
    )

set_a = pd.read_csv(SET_A_FILE)
set_b = pd.read_csv(SET_B_FILE)
set_c = pd.read_csv(SET_C_FILE)

print("Set A shape:", set_a.shape)
print("Set B shape:", set_b.shape)
print("Set C shape:", set_c.shape)

for name, df in [("A", set_a), ("B", set_b), ("C", set_c)]:
    if "in_hospital_death" in df.columns:
        print(f"Set {name} death rate: {df['in_hospital_death'].mean():.4f}")


Set A shape: (4000, 340)
Set B shape: (4000, 340)
Set C shape: (4000, 340)
Set A death rate: 0.1385
Set B death rate: 0.1420
Set C death rate: 0.1462


In [4]:
# ============================================================
# 3. Define target and columns to exclude
# ============================================================

TARGET = "in_hospital_death"

DROP_ALWAYS = [
    "RecordID",
    "in_hospital_death",
    "SAPS-I",
    "SOFA",
    "Length_of_stay",
    "Survival",
]

for required_col in [TARGET]:
    for name, df in [("Set A", set_a), ("Set B", set_b), ("Set C", set_c)]:
        if required_col not in df.columns:
            raise ValueError(f"{required_col} is missing from {name}.")

missing_drop_cols = [c for c in DROP_ALWAYS if c not in set_a.columns]
if missing_drop_cols:
    print("Warning: these DROP_ALWAYS columns were not found in Set A:", missing_drop_cols)

print("Target:", TARGET)
print("Always dropped columns:", [c for c in DROP_ALWAYS if c in set_a.columns])


Target: in_hospital_death
Always dropped columns: ['RecordID', 'in_hospital_death', 'SAPS-I', 'SOFA', 'Length_of_stay', 'Survival']


In [5]:
# ============================================================
# 4. Create A+B training set and C test set
# ============================================================

train_df = pd.concat([set_a, set_b], axis=0, ignore_index=True)
test_df = set_c.copy()

# Keep target separately.
y_train = train_df[TARGET].astype(int).copy()
y_test = test_df[TARGET].astype(int).copy()

# Drop leakage / non-feature columns.
existing_drop_cols_train = [c for c in DROP_ALWAYS if c in train_df.columns]
existing_drop_cols_test = [c for c in DROP_ALWAYS if c in test_df.columns]

X_train_raw = train_df.drop(columns=existing_drop_cols_train).copy()
X_test_raw = test_df.drop(columns=existing_drop_cols_test).copy()

# Keep only columns that exist in both train and test.
common_features = sorted(set(X_train_raw.columns).intersection(set(X_test_raw.columns)))
X_train_raw = X_train_raw[common_features].copy()
X_test_raw = X_test_raw[common_features].copy()

# Convert all features to numeric when possible.
X_train_raw = X_train_raw.apply(pd.to_numeric, errors="coerce")
X_test_raw = X_test_raw.apply(pd.to_numeric, errors="coerce")

print("Raw X_train shape:", X_train_raw.shape)
print("Raw X_test shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


Raw X_train shape: (8000, 334)
Raw X_test shape: (4000, 334)
y_train shape: (8000,)
y_test shape: (4000,)


In [6]:
# ============================================================
# 5. Measurement-level missingness rule using *_count columns
# ------------------------------------------------------------
# If base_count is 0 or NaN in >= 50% of A+B training patients,
# drop the full feature group for that base variable.
# ============================================================

MISSING_THRESHOLD = 0.50

count_cols = [c for c in X_train_raw.columns if c.endswith("_count")]

measurement_missing_summary = []
drop_measurement_bases = []

for count_col in count_cols:
    base = count_col[:-len("_count")]
    # Treat count NaN as 0 because missing count means no usable measurement count.
    count_values = X_train_raw[count_col].fillna(0)
    zero_or_missing_rate = (count_values <= 0).mean()
    measurement_missing_summary.append({
        "base_variable": base,
        "count_column": count_col,
        "zero_or_missing_count_rate_train_ab": zero_or_missing_rate,
        "drop_group": zero_or_missing_rate >= MISSING_THRESHOLD,
    })
    if zero_or_missing_rate >= MISSING_THRESHOLD:
        drop_measurement_bases.append(base)

measurement_missing_df = pd.DataFrame(measurement_missing_summary).sort_values(
    "zero_or_missing_count_rate_train_ab", ascending=False
)

# Drop all columns that belong to sparse measurement groups.
drop_measurement_group_cols = []
for base in drop_measurement_bases:
    members = [c for c in X_train_raw.columns if c == base or c.startswith(base + "_")]
    drop_measurement_group_cols.extend(members)

drop_measurement_group_cols = sorted(set(drop_measurement_group_cols))

print("Number of *_count columns found:", len(count_cols))
print("Number of measurement groups dropped by count==0/NaN >= 50%:", len(drop_measurement_bases))
print("Number of feature columns dropped from those groups:", len(drop_measurement_group_cols))

display(measurement_missing_df.head(20))


Number of *_count columns found: 36
Number of measurement groups dropped by count==0/NaN >= 50%: 10
Number of feature columns dropped from those groups: 90


,base_variable,count_column,zero_or_missing_count_rate_train_ab,drop_group
31,TroponinI,TroponinI_count,0.951750,True
6,Cholesterol,Cholesterol_count,0.919000,True
32,TroponinT,TroponinT_count,0.774375,True
27,RespRate,RespRate_count,0.721500,True
3,Albumin,Albumin_count,0.594250,True
0,ALP,ALP_count,0.577625,True
5,Bilirubin,Bilirubin_count,0.569250,True
1,ALT,ALT_count,0.568000,True
2,AST,AST_count,0.567375,True
28,SaO2,SaO2_count,0.556875,True


In [7]:
# ============================================================
# 6. Individual column missingness rule after group-level drop
# ------------------------------------------------------------
# Drop any remaining feature with NaN rate >= 50% in A+B training data.
# ============================================================

X_train_after_group = X_train_raw.drop(columns=drop_measurement_group_cols, errors="ignore")
X_test_after_group = X_test_raw.drop(columns=drop_measurement_group_cols, errors="ignore")

missing_rate_after_group = X_train_after_group.isna().mean().sort_values(ascending=False)
drop_individual_missing_cols = missing_rate_after_group[missing_rate_after_group >= MISSING_THRESHOLD].index.tolist()

X_train_filtered = X_train_after_group.drop(columns=drop_individual_missing_cols, errors="ignore")
X_test_filtered = X_test_after_group.drop(columns=drop_individual_missing_cols, errors="ignore")

print("Columns after measurement-group drop:", X_train_after_group.shape[1])
print("Individual columns dropped by NaN >= 50%:", len(drop_individual_missing_cols))
print("Final feature count before imputation:", X_train_filtered.shape[1])

if drop_individual_missing_cols:
    display(pd.DataFrame({
        "dropped_feature": drop_individual_missing_cols,
        "missing_rate_train_ab": missing_rate_after_group.loc[drop_individual_missing_cols].values,
    }).head(30))


Columns after measurement-group drop: 244
Individual columns dropped by NaN >= 50%: 1
Final feature count before imputation: 243


,dropped_feature,missing_rate_train_ab
0,Lactate_std,0.63


In [8]:
# ============================================================
# 7. Imputation
# ------------------------------------------------------------
# - Remaining *_count columns: NaN -> 0
# - Other numeric columns: NaN -> A+B training mean
# ============================================================

X_train_imputed = X_train_filtered.copy()
X_test_imputed = X_test_filtered.copy()

remaining_count_cols = [c for c in X_train_imputed.columns if c.endswith("_count")]
other_cols = [c for c in X_train_imputed.columns if c not in remaining_count_cols]

# Count columns: missing count means no measurement observed.
X_train_imputed[remaining_count_cols] = X_train_imputed[remaining_count_cols].fillna(0)
X_test_imputed[remaining_count_cols] = X_test_imputed[remaining_count_cols].fillna(0)

# Other columns: mean imputation using A+B training means only.
train_means = X_train_imputed[other_cols].mean(axis=0)

# Safety fallback: if a column mean is still NaN, fill it with 0.
# This should rarely happen after missingness filtering, but prevents runtime errors.
train_means = train_means.fillna(0)

X_train_imputed[other_cols] = X_train_imputed[other_cols].fillna(train_means)
X_test_imputed[other_cols] = X_test_imputed[other_cols].fillna(train_means)

print("Final X_train shape:", X_train_imputed.shape)
print("Final X_test shape:", X_test_imputed.shape)
print("Remaining missing in X_train:", int(X_train_imputed.isna().sum().sum()))
print("Remaining missing in X_test:", int(X_test_imputed.isna().sum().sum()))
print("Remaining count columns:", len(remaining_count_cols))


Final X_train shape: (8000, 243)
Final X_test shape: (4000, 243)
Remaining missing in X_train: 0
Remaining missing in X_test: 0
Remaining count columns: 26


In [9]:
# ============================================================
# 8. Save X/y outputs and preprocessing metadata
# ============================================================

TAG = "a_plus_b_train_c_test_countzero_missing50"

X_TRAIN_OUT = OUTPUT_DIR / f"X_train_{TAG}_imputed.csv"
Y_TRAIN_OUT = OUTPUT_DIR / f"y_train_{TAG}.csv"
X_TEST_OUT = OUTPUT_DIR / f"X_test_{TAG}_imputed.csv"
Y_TEST_OUT = OUTPUT_DIR / f"y_test_{TAG}.csv"

METADATA_OUT = OUTPUT_DIR / f"preprocessing_metadata_{TAG}.json"
MEASUREMENT_MISSING_OUT = OUTPUT_DIR / f"measurement_missing_summary_{TAG}.csv"
DROPPED_FEATURES_OUT = OUTPUT_DIR / f"dropped_features_{TAG}.csv"
TRAIN_MEANS_OUT = OUTPUT_DIR / f"train_means_for_imputation_{TAG}.csv"

X_train_imputed.to_csv(X_TRAIN_OUT, index=False)
y_train.to_frame(TARGET).to_csv(Y_TRAIN_OUT, index=False)
X_test_imputed.to_csv(X_TEST_OUT, index=False)
y_test.to_frame(TARGET).to_csv(Y_TEST_OUT, index=False)

measurement_missing_df.to_csv(MEASUREMENT_MISSING_OUT, index=False)
train_means.rename("train_mean_for_imputation").to_csv(TRAIN_MEANS_OUT, header=True)

dropped_features = sorted(set(drop_measurement_group_cols + drop_individual_missing_cols))
dropped_features_df = pd.DataFrame({
    "feature": dropped_features,
    "drop_reason": [
        "measurement_group_count_zero_or_missing_rate_ge_50" if f in set(drop_measurement_group_cols) else "individual_nan_rate_ge_50"
        for f in dropped_features
    ]
})
dropped_features_df.to_csv(DROPPED_FEATURES_OUT, index=False)

metadata = {
    "target": TARGET,
    "train_sets": ["Set A", "Set B"],
    "test_set": "Set C",
    "missing_threshold": MISSING_THRESHOLD,
    "drop_always": [c for c in DROP_ALWAYS if c in train_df.columns],
    "raw_train_shape": list(X_train_raw.shape),
    "raw_test_shape": list(X_test_raw.shape),
    "final_train_shape": list(X_train_imputed.shape),
    "final_test_shape": list(X_test_imputed.shape),
    "n_count_columns_found": len(count_cols),
    "n_measurement_groups_dropped": len(drop_measurement_bases),
    "dropped_measurement_bases": sorted(drop_measurement_bases),
    "n_measurement_group_feature_columns_dropped": len(drop_measurement_group_cols),
    "n_individual_missing_columns_dropped": len(drop_individual_missing_cols),
    "individual_missing_columns_dropped": drop_individual_missing_cols,
    "remaining_count_columns": remaining_count_cols,
    "imputation_rule": {
        "count_columns": "NaN filled with 0 after filtering",
        "other_numeric_columns": "NaN filled with A+B training mean",
    },
    "output_files": {
        "X_train": str(X_TRAIN_OUT),
        "y_train": str(Y_TRAIN_OUT),
        "X_test": str(X_TEST_OUT),
        "y_test": str(Y_TEST_OUT),
        "metadata": str(METADATA_OUT),
        "measurement_missing_summary": str(MEASUREMENT_MISSING_OUT),
        "dropped_features": str(DROPPED_FEATURES_OUT),
        "train_means": str(TRAIN_MEANS_OUT),
    }
}

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved output files:")
for key, path in metadata["output_files"].items():
    print(f"  {key}: {path}")


Saved output files:
  X_train: c:\Users\junse\Documents\research\IUBDC 2026\output\X_train_a_plus_b_train_c_test_countzero_missing50_imputed.csv
  y_train: c:\Users\junse\Documents\research\IUBDC 2026\output\y_train_a_plus_b_train_c_test_countzero_missing50.csv
  X_test: c:\Users\junse\Documents\research\IUBDC 2026\output\X_test_a_plus_b_train_c_test_countzero_missing50_imputed.csv
  y_test: c:\Users\junse\Documents\research\IUBDC 2026\output\y_test_a_plus_b_train_c_test_countzero_missing50.csv
  metadata: c:\Users\junse\Documents\research\IUBDC 2026\output\preprocessing_metadata_a_plus_b_train_c_test_countzero_missing50.json
  measurement_missing_summary: c:\Users\junse\Documents\research\IUBDC 2026\output\measurement_missing_summary_a_plus_b_train_c_test_countzero_missing50.csv
  dropped_features: c:\Users\junse\Documents\research\IUBDC 2026\output\dropped_features_a_plus_b_train_c_test_countzero_missing50.csv
  train_means: c:\Users\junse\Documents\research\IUBDC 2026\output\train_m

In [10]:
# ============================================================
# 9. Final sanity checks
# ============================================================

assert X_train_imputed.shape[0] == y_train.shape[0]
assert X_test_imputed.shape[0] == y_test.shape[0]
assert list(X_train_imputed.columns) == list(X_test_imputed.columns)
assert X_train_imputed.isna().sum().sum() == 0
assert X_test_imputed.isna().sum().sum() == 0

summary = pd.DataFrame([
    {
        "split": "train_A_plus_B",
        "n_rows": X_train_imputed.shape[0],
        "n_features": X_train_imputed.shape[1],
        "death_rate": y_train.mean(),
        "remaining_missing_values": int(X_train_imputed.isna().sum().sum()),
    },
    {
        "split": "test_C",
        "n_rows": X_test_imputed.shape[0],
        "n_features": X_test_imputed.shape[1],
        "death_rate": y_test.mean(),
        "remaining_missing_values": int(X_test_imputed.isna().sum().sum()),
    },
])

display(summary)


,split,n_rows,n_features,death_rate,remaining_missing_values
0,train_A_plus_B,8000,243,0.14025,0
1,test_C,4000,243,0.14625,0


## Notes for later modeling

For **Logistic Regression**, use a pipeline with `StandardScaler` after this preprocessing because the features have very different scales.

For **XGBoost**, scaling is not necessary. Because ICU mortality is class-imbalanced, consider using `scale_pos_weight = number_of_survivors / number_of_deaths` computed from `y_train`.

Set C should remain a held-out test set. Do not use Set C for feature selection, hyperparameter tuning, threshold tuning, or explanation metric design.
